# Phase 1: Image Filter Kernels
## Gaussian · Sobel · Haar-like

This notebook demonstrates hand-crafted convolution kernels rebuilt in NumPy,
replacing the original MATLAB implementations (`conv2d.m`, `Gaussian.m`, `Sobel.m`, `Haarlike.m`).

**Key improvements over the original MATLAB:**
- `np.kron` replaces the nested-loop `set_scale.m`
- Unified `apply_*` interface that handles both grayscale and RGB inputs
- Interactive visualization of kernel values and filter outputs

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from scipy import datasets

from src.filters import (
    apply_gaussian, apply_sobel, apply_haar,
    gaussian_kernel, sobel_kernels, haar_kernel,
    set_scale, conv2d_manual
)

rcParams['figure.dpi'] = 120
rcParams['font.size'] = 11
print('✅ All imports OK')

## 1. Load Test Image

In [ ]:
image = datasets.ascent()
print(f'Shape: {image.shape}')
print(f'Value range: [{image.min()}, {image.max()}]')

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(image, cmap='gray')
ax.set_title('Original Image (scipy.datasets.ascent)', fontsize=14)
ax.axis('off')
plt.show()

## 2. Kernel Visualization

Let's look at the raw kernels before applying them.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

# Gaussian 3x3
gk = gaussian_kernel(3)
im0 = axes[0].imshow(gk, cmap='Blues')
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, f'{gk[i,j]:.3f}', ha='center', va='center', fontsize=9)
axes[0].set_title('Gaussian 3×3', fontsize=13)
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# Sobel Gx
Gx, Gy = sobel_kernels()
im1 = axes[1].imshow(Gx, cmap='RdBu', vmin=-2, vmax=2)
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, f'{Gx[i,j]:.0f}', ha='center', va='center', fontsize=9)
axes[1].set_title('Sobel Gx', fontsize=13)
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# Haar12
h12 = haar_kernel('Haar12')
im2 = axes[2].imshow(h12, cmap='RdBu', vmin=-1, vmax=1)
for i in range(1):
    for j in range(2):
        axes[2].text(j, i, f'{h12[i,j]:.0f}', ha='center', va='center', fontsize=11)
axes[2].set_title('Haar12', fontsize=13)
plt.colorbar(im2, ax=axes[2], fraction=0.046)

# Haar22
h22 = haar_kernel('Haar22')
im3 = axes[3].imshow(h22, cmap='RdBu', vmin=-1, vmax=1)
for i in range(2):
    for j in range(2):
        axes[3].text(j, i, f'{h22[i,j]:.0f}', ha='center', va='center', fontsize=11)
axes[3].set_title('Haar22', fontsize=13)
plt.colorbar(im3, ax=axes[3], fraction=0.046)

plt.suptitle('Kernel Visualization', fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

## 3. Kernel Scaling (`set_scale`)

The original MATLAB `set_scale.m` uses nested loops to replicate each kernel element.
Here we use `np.kron` (Kronecker product) which does the same thing in one line.

$$\text{Gaussian}_{3×3} \xrightarrow{\text{scale}=4} \text{Gaussian}_{12×12}$$

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for idx, (scale, ax) in enumerate(zip([1, 2, 4], axes)):
    kernel = haar_kernel('Haar22', scale=scale)
    ax.imshow(kernel, cmap='RdBu', vmin=-1, vmax=1)
    h, w = kernel.shape
    ax.set_title(f'Haar22 scale={scale}\n({h}×{w})', fontsize=12)
    ax.axis('off')

plt.suptitle('set_scale: Kronecker Product Scaling', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Gaussian Smoothing

Increasing kernel size → stronger smoothing effect

In [ ]:
kern_sizes = [3, 5, 7, 11]
fig, axes = plt.subplots(1, len(kern_sizes) + 1, figsize=(16, 4))

axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original', fontsize=13)
axes[0].axis('off')

for idx, k in enumerate(kern_sizes):
    result = apply_gaussian(image, kernel_size=k, scale=1)
    axes[idx+1].imshow(result, cmap='gray')
    axes[idx+1].set_title(f'Gaussian k={k}', fontsize=13)
    axes[idx+1].axis('off')

plt.suptitle('Gaussian Smoothing at Different Kernel Sizes', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Sobel Edge Detection

$$\text{Gradient Magnitude} = \sqrt{G_x^2 + G_y^2}$$

Scaling the kernel → thicker edges

In [ ]:
scales = [1, 2, 4, 8]
fig, axes = plt.subplots(1, len(scales) + 1, figsize=(16, 4))

axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original', fontsize=13)
axes[0].axis('off')

for idx, s in enumerate(scales):
    result = apply_sobel(image, scale=s)
    axes[idx+1].imshow(result, cmap='gray')
    axes[idx+1].set_title(f'Sobel scale={s}', fontsize=13)
    axes[idx+1].axis('off')

plt.suptitle('Sobel Edge Detection at Different Scales', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. All Five Haar-like Features

Haar-like kernels detect local intensity patterns: edges, lines, and center-surround features.

In [ ]:
haar_types = ['Haar12', 'Haar21', 'Haar13', 'Haar31', 'Haar22']
haar_labels = ['Horizontal 2-rect', 'Vertical 2-rect', 'Horizontal 3-rect',
               'Vertical 3-rect', 'Checkerboard']

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for idx, (ht, label) in enumerate(zip(haar_types, haar_labels)):
    # Top row: kernel
    kernel = haar_kernel(ht, scale=4)
    axes[0, idx].imshow(kernel, cmap='RdBu', vmin=-1, vmax=1)
    axes[0, idx].set_title(f'{ht}\n{label}', fontsize=10)
    axes[0, idx].axis('off')

    # Bottom row: filter response
    result = apply_haar(image, kernel_type=ht, scale=4)
    vmax = np.abs(result).max()
    axes[1, idx].imshow(result, cmap='RdBu', vmin=-vmax, vmax=vmax)
    axes[1, idx].set_title('Response', fontsize=10)
    axes[1, idx].axis('off')

axes[0, 0].set_ylabel('Kernel', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Response', fontsize=13, fontweight='bold')
plt.suptitle('Haar-like Feature Kernels & Responses (scale=4)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Summary: MATLAB vs Python

| Feature | Original MATLAB | This Python Implementation |
|---------|----------------|---------------------------|
| 2D Convolution | Nested loops in `conv2d.m` | `scipy.signal.convolve2d` |
| Kernel Scaling | Nested loops in `set_scale.m` | `np.kron` (single line) |
| Gaussian | Hardcoded 3×3 in `Gaussian.m` | Generic size support |
| Sobel | `Sobel.m` | Same math, vectorized |
| Haar-like | `Haarlike.m` with string dispatch | Dictionary lookup + type hints |
| Color Handling | `rgb2gray` required | Auto-convert via `_to_grayscale` |

---
**Next: Phase 2 — SIFT Feature Extraction & Matching**